In [20]:
# ----------------------------------
# Aurora Project: Fine-Tuning Modified Advanced Autoencoder on Multiple CDF Data Sets
# ----------------------------------

import os
import datetime
import numpy as np
import cdflib
from scipy.interpolate import interp1d
import tensorflow as tf
from tensorflow.keras.layers import (
    Conv3D, MaxPooling3D, Reshape, LSTM, Dense, BatchNormalization,
    Dropout, Attention, GlobalAveragePooling1D, TimeDistributed
)
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import random

# ----------------------------------
# 1. Set Random Seeds for Reproducibility
# ----------------------------------
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

# ----------------------------------
# 2. Define Paths and Helper Functions
# ----------------------------------

# Paths to the data folders (Update these paths as per your directory structure)
fpi_dist_folder = r"C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data\mms1\fpi\brst\l2\des-dist\2024\03\01"
fgm_folder = r"C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data\mms1\fgm"
fpi_moms_folder = r"C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data\mms1\fpi\brst\l2\des-moms\2024\03\01"

def extract_timestamp(filename):
    """
    Extract timestamp from filename.
    Assumes the timestamp is a 14-digit number (YYYYMMDDHHMMSS) part of the filename.

    Example filename: 'mms1_fpi_brst_l2_des-dist_20240301230743_v3.4.0.cdf'
    """
    try:
        parts = filename.split('_')
        for part in parts:
            if part.isdigit() and len(part) == 14:
                return datetime.datetime.strptime(part, "%Y%m%d%H%M%S")
        raise ValueError("No valid timestamp found in filename")
    except Exception as e:
        print(f"Error extracting timestamp from filename: {filename} - {e}")
        return None

def load_fgm_data(filepath):
    """Load magnetic field data from FGM CDF file."""
    with cdflib.CDF(filepath) as fgm_cdf:
        epoch = fgm_cdf.varget('Epoch')  # Time variable for FGM data
        b_gse = fgm_cdf.varget('mms1_fgm_b_gse_brst_l2')  # Magnetic field in GSE coordinates
    return epoch, b_gse

def load_fpi_dist_data(filepath):
    """Load distribution data from FPI (`des-dist`) CDF file."""
    with cdflib.CDF(filepath) as fpi_cdf:
        epoch = fpi_cdf.varget('Epoch')  # Time variable for FPI distribution data
        dist_data = fpi_cdf.varget('mms1_des_dist_brst')  # Distribution function data
        theta = fpi_cdf.varget('mms1_des_theta_brst')  # Theta angles
        phi = fpi_cdf.varget('mms1_des_phi_brst')  # Phi angles
        energy = fpi_cdf.varget('mms1_des_energy_brst')  # Energy levels
    return epoch, dist_data, theta, phi, energy  # Return energy

def load_fpi_moms_data(filepath):
    """Load bulk velocity data from FPI (`des-moms`) CDF file."""
    with cdflib.CDF(filepath) as moms_cdf:
        epoch = moms_cdf.varget('Epoch')  # Time variable for FPI moments data
        bulk_velocity = moms_cdf.varget('mms1_des_bulkv_gse_brst')  # Bulk velocity in GSE coordinates
    return epoch, bulk_velocity

def interpolate_b_field(fgm_epoch, b_gse, fpi_epoch):
    """Interpolate magnetic field data to match FPI timestamps."""
    # Extract only the first three components (x, y, z) from b_gse to match bulk velocity shape
    b_gse = b_gse[:, :3]
    interpolator = interp1d(fgm_epoch, b_gse, axis=0, bounds_error=False, fill_value="extrapolate")
    return interpolator(fpi_epoch)

def calculate_pitch_angle(phi_bins, theta_bins, b_field):
    """Calculate pitch angle distribution using look direction (phi, theta) and magnetic field."""
    # Convert theta and phi to radians
    theta_rad = np.deg2rad(theta_bins)  # Shape: (theta_bins,)
    phi_rad = np.deg2rad(phi_bins)      # Shape: (phi_bins,)

    # Create meshgrid for phi and theta
    phi_grid, theta_grid = np.meshgrid(phi_rad, theta_rad, indexing='ij')  # Shapes: (phi_bins, theta_bins)

    # Compute look direction unit vector components
    look_direction_x = np.sin(theta_grid) * np.cos(phi_grid)
    look_direction_y = np.sin(theta_grid) * np.sin(phi_grid)
    look_direction_z = np.cos(theta_grid)

    # Stack components to form the look direction vectors
    look_direction = np.stack(
        [look_direction_x, look_direction_y, look_direction_z],
        axis=2
    )  # Shape: (phi_bins, theta_bins, 3)

    # Flatten look_direction for efficient computation
    look_direction_flat = look_direction.reshape(-1, 3)  # Shape: (phi_bins * theta_bins, 3)

    # Initialize the pitch angles array
    time_steps = b_field.shape[0]
    pitch_angles = np.zeros((time_steps, phi_bins.size, theta_bins.size))  # Shape: (time_steps, phi_bins, theta_bins)

    # Iterate over each time step
    for t in range(time_steps):
        # Normalize magnetic field vector
        b_vector = b_field[t, :3]  # Assuming b_field has shape (time_steps, 3)
        b_magnitude = np.linalg.norm(b_vector)
        if b_magnitude == 0:
            b_unit = np.zeros(3)
        else:
            b_unit = b_vector / b_magnitude

        # Compute dot product between look direction and magnetic field
        v_dot_b = np.dot(look_direction_flat, b_unit)  # Shape: (phi_bins * theta_bins,)
        v_dot_b = np.clip(v_dot_b, -1.0, 1.0)  # Ensure values are within valid range

        # Calculate pitch angles in degrees
        pitch_angles_flat = np.arccos(v_dot_b) * (180.0 / np.pi)

        # Reshape back to (phi_bins, theta_bins)
        pitch_angles[t] = pitch_angles_flat.reshape(phi_bins.size, theta_bins.size)

    return pitch_angles  # Shape: (time_steps,32,16)

def match_files(fpi_dist_folder, fgm_folder, fpi_moms_folder):
    """
    Match distribution, magnetic field, and moments CDF files based on their timestamps.

    Returns:
    - matched_files (list of tuples): Each tuple contains (timestamp, fpi_dist_file, fgm_file, fpi_moms_file)
    """
    matched_files = []
    fpi_dist_files = [f for f in os.listdir(fpi_dist_folder) if f.endswith('.cdf')]

    for fpi_dist_filename in fpi_dist_files:
        fpi_dist_timestamp = extract_timestamp(fpi_dist_filename)
        if fpi_dist_timestamp is None:
            continue

        # Find matching FGM file
        fgm_files = [f for f in os.listdir(fgm_folder) if f.endswith('.cdf')]
        matching_fgm = [f for f in fgm_files if extract_timestamp(f) == fpi_dist_timestamp]
        if not matching_fgm:
            print(f"No matching FGM file for timestamp {fpi_dist_timestamp}")
            continue
        fgm_filename = matching_fgm[0]

        # Find matching FPI Moments file
        fpi_moms_files = [f for f in os.listdir(fpi_moms_folder) if f.endswith('.cdf')]
        matching_moms = [f for f in fpi_moms_files if extract_timestamp(f) == fpi_dist_timestamp]
        if not matching_moms:
            print(f"No matching FPI Moments file for timestamp {fpi_dist_timestamp}")
            continue
        fpi_moms_filename = matching_moms[0]

        matched_files.append((fpi_dist_timestamp, fpi_dist_filename, fgm_filename, fpi_moms_filename))

    return matched_files

# ----------------------------------
# 3. Define Data Preparation Function
# ----------------------------------

def prepare_data(fgm_filepath, fpi_dist_filepath, fpi_moms_filepath):
    """
    Load and preprocess data from a single matched CDF file set.
    
    Parameters:
    - fgm_filepath: Path to the FGM CDF file.
    - fpi_dist_filepath: Path to the FPI Distribution CDF file.
    - fpi_moms_filepath: Path to the FPI Moments CDF file.
    
    Returns:
    - X_data: Preprocessed input data ready for model input. Shape: (time_steps, 1, 32, 16, 33)
    - y_data: Corresponding target data for training. Shape: (time_steps,32,16,32)
    """
    # Load FGM data
    fgm_epoch, b_gse = load_fgm_data(fgm_filepath)
    print(f"Loaded FGM data from {fgm_filepath}.")
    
    # Load FPI Distribution data
    fpi_epoch, dist_data, theta, phi, energy = load_fpi_dist_data(fpi_dist_filepath)
    print(f"Loaded FPI Distribution data from {fpi_dist_filepath}.")
    
    # Load FPI Moments data
    moms_epoch, bulk_velocity = load_fpi_moms_data(fpi_moms_filepath)
    print(f"Loaded FPI Moments data from {fpi_moms_filepath}.")
    
    # Interpolate magnetic field to match FPI timestamps
    b_gse_interp = interpolate_b_field(fgm_epoch, b_gse, fpi_epoch)
    print("Interpolated magnetic field data to match FPI timestamps.")
    
    # Prepare phi_bins and theta_bins
    if phi.ndim == 1:
        phi_bins = phi.flatten()  # Shape: (phi_bins=32,)
    elif phi.ndim == 2:
        phi_bins = phi[0].flatten()
        print("Phi has an extra time dimension. Extracted phi from the first time step.")
    else:
        raise ValueError("Unexpected shape for phi_bins.")
    
    if theta.ndim == 1:
        theta_bins = theta.flatten()  # Shape: (theta_bins=16,)
    elif theta.ndim == 2:
        theta_bins = theta[0].flatten()
        print("Theta has an extra time dimension. Extracted theta from the first time step.")
    else:
        raise ValueError("Unexpected shape for theta_bins.")
    
    # Verify shapes
    print(f"Phi bins shape: {phi_bins.shape}")     # Expected: (32,)
    print(f"Theta bins shape: {theta_bins.shape}") # Expected: (16,)
    
    # Compute pitch angles using the provided function
    pitch_angles = calculate_pitch_angle(phi_bins, theta_bins, b_gse_interp)
    print("Calculated pitch angles.")
    
    # Do not transpose pitch_angles; keep (time_steps, phi_bins, theta_bins)
    print(f"pitch_angles shape: {pitch_angles.shape}")  # Should be (time_steps,32,16)
    
    # Normalize pitch angles to [0, 1] for input to the model
    pitch_angles_normalized = pitch_angles / 180.0  # Shape: (time_steps,32,16)
    
    # Transpose distribution data to (time_steps, phi_bins, theta_bins, energy_levels)
    # Original shape assumed: (time_steps, energy_levels=32, phi_bins=32, theta_bins=16)
    # Desired shape: (time_steps, phi_bins=32, theta_bins=16, energy_levels=32)
    
    # Verify current shape
    if dist_data.ndim != 4:
        raise ValueError("Distribution data should have 4 dimensions (time_steps, energy_levels, phi_bins, theta_bins).")
    
    # Transpose if necessary
    if dist_data.shape[1] == energy.size and dist_data.shape[2] == phi_bins.size and dist_data.shape[3] == theta_bins.size:
        dist_data_transposed = dist_data.transpose(0, 2, 3, 1)  # (time_steps, phi_bins, theta_bins, energy_levels)
        print("Transposed distribution data to shape (time_steps, phi_bins, theta_bins, energy_levels).")
    else:
        # If already in desired shape, no transpose
        dist_data_transposed = dist_data
        print("Distribution data is already in the desired shape.")
    
    print(f"Transposed Distribution Data Shape: {dist_data_transposed.shape}")  # Should be (time_steps,32,16,32)
    
    # Ensure that pitch_angles and dist_data_transposed have matching phi_bins and theta_bins
    if pitch_angles_normalized.shape[1] != dist_data_transposed.shape[1] or \
       pitch_angles_normalized.shape[2] != dist_data_transposed.shape[2]:
        raise ValueError("Mismatch in phi_bins or theta_bins dimensions between pitch_angles and dist_data_transposed.")
    
    # Expand pitch_angles to add a new feature (energy_levels +1)
    pitch_angles_expanded = pitch_angles_normalized[..., np.newaxis]  # Shape: (time_steps,32,16,1)
    print(f"Pitch Angles Expanded Shape: {pitch_angles_expanded.shape}")  # Shape: (time_steps,32,16,1)
    
    # Concatenate distribution data and pitch angles along the energy_levels axis
    X_data = np.concatenate((
        dist_data_transposed,
        pitch_angles_expanded
    ), axis=3)  # Shape: (time_steps,32,16,33)
    X_data = X_data.astype('float32')
    print(f"Input Data Shape with Pitch Angles: {X_data.shape}")  # Should be (time_steps,32,16,33)
    
    # Prepare targets (PSD data without pitch angles)
    y_data = dist_data_transposed  # Shape: (time_steps,32,16,32)
    
    # Reshape data to include time_steps=1 for model input
    # Current X_data shape: (time_steps,32,16,33)
    # Desired shape: (time_steps,1,32,16,33)
    X_data = X_data.reshape((X_data.shape[0], 1, X_data.shape[1], X_data.shape[2], X_data.shape[3]))  # (time_steps,1,32,16,33)
    print(f"Reshaped Input Data Shape: {X_data.shape}")  # (time_steps,1,32,16,33)
    
    # Targets shape: (time_steps,32,16,32)
    # For model training, targets should match model output shape: (time_steps,32,16,32)
    # No need to reshape y_data
    print(f"Target Data Shape: {y_data.shape}")  # (time_steps,32,16,32)
    
    return X_data, y_data

# ----------------------------------
# 4. Define Freeze Function
# ----------------------------------

def freeze_all_but_last_layers(model, num_conv_layers_to_keep=1, num_lstm_layers_to_keep=1):
    """
    Freeze all Conv3D and LSTM layers except the last `num_conv_layers_to_keep` Conv3D layers
    and the last `num_lstm_layers_to_keep` LSTM layers.

    Parameters:
    - model: Keras model to modify.
    - num_conv_layers_to_keep: Number of Conv3D layers from the end to keep trainable.
    - num_lstm_layers_to_keep: Number of LSTM layers from the end to keep trainable.

    Returns:
    - None
    """
    # Identify Conv3D and LSTM layers
    conv3d_layers = [layer for layer in model.layers if isinstance(layer, Conv3D)]
    lstm_layers = [layer for layer in model.layers if isinstance(layer, LSTM)]
    
    print(f"Total Conv3D layers: {len(conv3d_layers)}")
    print(f"Total LSTM layers: {len(lstm_layers)}")
    
    # Freeze all Conv3D layers except the last `num_conv_layers_to_keep`
    if num_conv_layers_to_keep < len(conv3d_layers):
        layers_to_freeze = conv3d_layers[:-num_conv_layers_to_keep]
    else:
        layers_to_freeze = []
    
    for layer in layers_to_freeze:
        layer.trainable = False
        print(f"Frozen Conv3D layer: {layer.name}")
    
    # Freeze all LSTM layers except the last `num_lstm_layers_to_keep`
    if num_lstm_layers_to_keep < len(lstm_layers):
        layers_to_freeze = lstm_layers[:-num_lstm_layers_to_keep]
    else:
        layers_to_freeze = []
    
    for layer in layers_to_freeze:
        layer.trainable = False
        print(f"Frozen LSTM layer: {layer.name}")
    
    # Optionally, you can print the trainable status of all layers
    for layer in model.layers:
        print(f"Layer {layer.name} trainable: {layer.trainable}")

# ----------------------------------
# 5. Load the Modified Model
# ----------------------------------

# Path to the modified model (without Lambda layer)
modified_model_path = r"C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\model.h5"

# Verify that the modified model file exists
if not os.path.exists(modified_model_path):
    raise FileNotFoundError(f"Modified model file not found at: {modified_model_path}")
else:
    print("Modified model file found. Proceeding to load.")

    # Load the modified model
    try:
        modified_model = load_model(modified_model_path)
        print("Modified model loaded successfully.")
        modified_model.summary()
    except Exception as e:
        raise ValueError(f"Error loading the modified model: {e}")

# ----------------------------------
# 6. Freeze Specific Layers
# ----------------------------------

print("\nFreezing all Conv3D and LSTM layers except the last 1 Conv3D layer and the last 1 LSTM layer...")
freeze_all_but_last_layers(modified_model, num_conv_layers_to_keep=1, num_lstm_layers_to_keep=1)

# ----------------------------------
# 7. Compile the Model After Freezing Layers
# ----------------------------------

# It's crucial to compile the model after setting layer.trainable
modified_model.compile(optimizer="adam", 
                       loss='mean_squared_error')
print("\nModel compiled for fine-tuning with a lower learning rate.")

# ----------------------------------
# 8. Fine-Tuning Preparation
# ----------------------------------

# Perform file matching using the existing match_files function
matched_files = match_files(fpi_dist_folder, fgm_folder, fpi_moms_folder)

# Check if any matched files exist
if not matched_files:
    raise FileNotFoundError("No matched file sets found.")

# ----------------------------------
# 9. Fine-Tuning Function (Updated)
# ----------------------------------

def fine_tune_model_on_multiple_cdf_sets(
    matched_files,
    fpi_dist_folder,
    fgm_folder,
    fpi_moms_folder,
    model,
    fine_tune_epochs=10,
    batch_size=32,
    checkpoint_dir='fine_tuned_models'
):
    """
    Fine-tune the pre-trained model on multiple matched CDF file sets.
    
    Parameters:
    - matched_files: List of tuples containing matched CDF file names.
    - fpi_dist_folder: Directory containing FPI Distribution CDF files.
    - fgm_folder: Directory containing FGM CDF files.
    - fpi_moms_folder: Directory containing FPI Moments CDF files.
    - model: Pre-trained Keras model to be fine-tuned.
    - fine_tune_epochs: Number of epochs for fine-tuning on each file set.
    - batch_size: Batch size for training.
    - checkpoint_dir: Directory to save the fine-tuned models.
    
    Returns:
    - None
    """
    # Ensure the checkpoint directory exists
    os.makedirs(checkpoint_dir, exist_ok=True)
    print(f"Checkpoint directory set to: {checkpoint_dir}")
    
    # Fine-tune on all matched file sets
    total_sets = len(matched_files)
    print(f"\nTotal matched file sets to fine-tune on: {total_sets}")
    
    for idx, (timestamp, fpi_dist_filename, fgm_filename, fpi_moms_filename) in enumerate(matched_files, start=1):
        print(f"\n--- Fine-Tuning on CDF File Set {idx}/{total_sets} ---")
        print(f"Timestamp: {timestamp}")
        print(f"FPI Distribution File: {fpi_dist_filename}")
        print(f"FGM File: {fgm_filename}")
        print(f"FPI Moments File: {fpi_moms_filename}")
        
        # Construct full file paths
        fpi_dist_filepath = os.path.join(fpi_dist_folder, fpi_dist_filename)
        fgm_filepath = os.path.join(fgm_folder, fgm_filename)
        fpi_moms_filepath = os.path.join(fpi_moms_folder, fpi_moms_filename)
        
        # Prepare data
        X_data, y_data = prepare_data(fgm_filepath, fpi_dist_filepath, fpi_moms_filepath)
        
        # Split into fine-tuning training and validation sets (e.g., 80% train, 20% val)
        total_time_steps = X_data.shape[0]
        n_finetune_train = int(0.8 * total_time_steps)
        k_finetune_val = total_time_steps - n_finetune_train
        
        X_finetune_train = X_data[:n_finetune_train]
        y_finetune_train = y_data[:n_finetune_train]
        
        X_finetune_val = X_data[n_finetune_train:]
        y_finetune_val = y_data[n_finetune_train:]
        
        print(f"Fine-Tuning Training Data Shape: {X_finetune_train.shape}")  # (n_finetune_train,1,32,16,33)
        print(f"Fine-Tuning Validation Data Shape: {X_finetune_val.shape}")  # (k_finetune_val,1,32,16,33)
        
        # Define callbacks
        early_stopping = EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True,
            verbose=1
        )
        
        # Define a unique checkpoint path for each set
        checkpoint_path = os.path.join(checkpoint_dir, f'fine_tuned_model_set_{idx}.h5')
        checkpoint = ModelCheckpoint(
            filepath=checkpoint_path,
            monitor='val_loss',
            save_best_only=True,
            save_weights_only=False,  # Ensures the entire model is saved
            mode='min',
            verbose=1
        )
        
        # Fine-Tune the Model on Current CDF File Set
        history = model.fit(
            X_finetune_train,
            y_finetune_train,
            epochs=fine_tune_epochs,
            batch_size=batch_size,
            shuffle=False,  # Preserve time dependency
            validation_data=(X_finetune_val, y_finetune_val),
            callbacks=[early_stopping, checkpoint]
        )
        
        print(f"Fine-tuning on CDF File Set {idx} completed.")
        print(f"Best model for set {idx} saved at: {checkpoint_path}")
        
        # Optionally, save the training history for later analysis
        history_path = os.path.join(checkpoint_dir, f'history_finetune_set_{idx}.npy')
        np.save(history_path, history.history)
        print(f"Training history saved at: {history_path}")
    
    print("\nAll fine-tuning steps completed.")


Modified model file found. Proceeding to load.
Modified model loaded successfully.
Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_layer (InputLayer)    [(None, None, 32, 16, 33)]   0         []                            
                                                                                                  
 conv3d_4 (Conv3D)           (None, None, 32, 16, 64)     57088     ['input_layer[0][0]']         
                                                                                                  
 batch_normalization_8 (Bat  (None, None, 32, 16, 64)     256       ['conv3d_4[0][0]']            
 chNormalization)                                                                                 
                                                                                                  
 max_poolin

Total params: 13118848 (50.04 MB)
Trainable params: 9345792 (35.65 MB)
Non-trainable params: 3773056 (14.39 MB)
__________________________________________________________________________________________________

Freezing all Conv3D and LSTM layers except the last 1 Conv3D layer and the last 1 LSTM layer...
Total Conv3D layers: 4
Total LSTM layers: 3
Frozen Conv3D layer: conv3d_4
Frozen Conv3D layer: conv3d_5
Frozen Conv3D layer: conv3d_6
Frozen LSTM layer: lstm_3
Frozen LSTM layer: lstm_4
Layer input_layer trainable: True
Layer conv3d_4 trainable: False
Layer batch_normalization_8 trainable: True
Layer max_pooling3d_4 trainable: True
Layer conv3d_5 trainable: False
Layer batch_normalization_9 trainable: True
Layer max_pooling3d_5 trainable: True
Layer conv3d_6 trainable: False
Layer batch_normalization_10 trainable: True
Layer max_pooling3d_6 trainable: True
Layer conv3d_7 trainable: False
Layer batch_normalization_11 trainable: True
Layer max_pooling3d_7 trainable: True
Layer time_dis

In [21]:
# Path to the modified model (without Lambda layer)
modified_model_path = r"C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\model.h5"

# Verify that the modified model file exists
if not os.path.exists(modified_model_path):
    raise FileNotFoundError(f"Modified model file not found at: {modified_model_path}")
else:
    print("Modified model file found. Proceeding to load.")

    # Load the modified model
    try:
        modified_model = load_model(modified_model_path)
        print("Modified model loaded successfully.")
        modified_model.summary()
    except Exception as e:
        raise ValueError(f"Error loading the modified model: {e}")

print("\nFreezing all Conv3D and LSTM layers except the last 1 Conv3D layer and the last 1 LSTM layer...")
freeze_all_but_last_layers(modified_model, num_conv_layers_to_keep=1, num_lstm_layers_to_keep=1)

# Compile the model after freezing layers
modified_model.compile(optimizer="adam", 
                       loss='mean_squared_error')


Modified model file found. Proceeding to load.
Modified model loaded successfully.
Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_layer (InputLayer)    [(None, None, 32, 16, 33)]   0         []                            
                                                                                                  
 conv3d_4 (Conv3D)           (None, None, 32, 16, 64)     57088     ['input_layer[0][0]']         
                                                                                                  
 batch_normalization_8 (Bat  (None, None, 32, 16, 64)     256       ['conv3d_4[0][0]']            
 chNormalization)                                                                                 
                                                                                                  
 max_poolin

Total params: 13118848 (50.04 MB)
Trainable params: 9345792 (35.65 MB)
Non-trainable params: 3773056 (14.39 MB)
__________________________________________________________________________________________________

Freezing all Conv3D and LSTM layers except the last 1 Conv3D layer and the last 1 LSTM layer...
Total Conv3D layers: 4
Total LSTM layers: 3
Frozen Conv3D layer: conv3d_4
Frozen Conv3D layer: conv3d_5
Frozen Conv3D layer: conv3d_6
Frozen LSTM layer: lstm_3
Frozen LSTM layer: lstm_4
Layer input_layer trainable: True
Layer conv3d_4 trainable: False
Layer batch_normalization_8 trainable: True
Layer max_pooling3d_4 trainable: True
Layer conv3d_5 trainable: False
Layer batch_normalization_9 trainable: True
Layer max_pooling3d_5 trainable: True
Layer conv3d_6 trainable: False
Layer batch_normalization_10 trainable: True
Layer max_pooling3d_6 trainable: True
Layer conv3d_7 trainable: False
Layer batch_normalization_11 trainable: True
Layer max_pooling3d_7 trainable: True
Layer time_dis

In [22]:
# Perform file matching using the existing match_files function
matched_files = match_files(fpi_dist_folder, fgm_folder, fpi_moms_folder)

# Check if any matched files exist
if not matched_files:
    raise FileNotFoundError("No matched file sets found.")

# Fine-tune the modified model on all matched file sets
fine_tune_model_on_multiple_cdf_sets(
    matched_files=matched_files,
    fpi_dist_folder=fpi_dist_folder,
    fgm_folder=fgm_folder,
    fpi_moms_folder=fpi_moms_folder,
    model=modified_model,  # Ensure this refers to your loaded modified model
    fine_tune_epochs=10,    # Adjust as needed
    batch_size=32,          # Adjust as needed
    checkpoint_dir='fine_tuned_models'  # Adjust as needed
)


No matching FPI Moments file for timestamp 2024-03-01 23:22:33
No matching FPI Moments file for timestamp 2024-03-01 23:29:13
Checkpoint directory set to: fine_tuned_models

Total matched file sets to fine-tune on: 5

--- Fine-Tuning on CDF File Set 1/5 ---
Timestamp: 2024-03-01 23:07:43
FPI Distribution File: mms1_fpi_brst_l2_des-dist_20240301230743_v3.4.0.cdf
FGM File: mms1_fgm_brst_l2_20240301230743_v5.441.0.cdf
FPI Moments File: mms1_fpi_brst_l2_des-moms_20240301230743_v3.4.0.cdf
Loaded FGM data from C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data\mms1\fgm\mms1_fgm_brst_l2_20240301230743_v5.441.0.cdf.
Loaded FPI Distribution data from C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data\mms1\fpi\brst\l2\des-dist\2024\03\01\mms1_fpi_brst_l2_des-dist_20240301230743_v3.4.0.cdf.
Loaded FPI Moments data from C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data\mms1\fpi\brst\l2\des-moms\2024\03\01\mms1_fpi_brst_l2_des-moms_20240301230743_v3.4.0.cdf.
Interpola

C:\Users\ryane\anaconda3\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


108/109 [============================>.] - ETA: 0s - loss: 0.0000e+00
Epoch 2: val_loss improved from 0.00000 to 0.00000, saving model to fine_tuned_models\fine_tuned_model_set_1.h5
109/109 [==============================] - 6s 54ms/step - loss: 0.0000e+00 - val_loss: 2.6073e-26
Epoch 3/10
109/109 [==============================] - ETA: 0s - loss: 1.6270e-21
Epoch 3: val_loss did not improve from 0.00000
109/109 [==============================] - 6s 51ms/step - loss: 1.6270e-21 - val_loss: 4.7967e-25
Epoch 4/10
109/109 [==============================] - ETA: 0s - loss: 1.2714e-13
Epoch 4: val_loss did not improve from 0.00000
109/109 [==============================] - 6s 53ms/step - loss: 1.2714e-13 - val_loss: 2.3364e-17
Epoch 5/10
108/109 [============================>.] - ETA: 0s - loss: 4.7436e-22
Epoch 5: val_loss did not improve from 0.00000
109/109 [==============================] - 6s 53ms/step - loss: 4.7299e-22 - val_loss: 7.4453e-07
Epoch 6/10
109/109 [======================

Loaded FPI Distribution data from C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data\mms1\fpi\brst\l2\des-dist\2024\03\01\mms1_fpi_brst_l2_des-dist_20240301232443_v3.4.0.cdf.
Loaded FPI Moments data from C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data\mms1\fpi\brst\l2\des-moms\2024\03\01\mms1_fpi_brst_l2_des-moms_20240301232443_v3.4.0.cdf.
Interpolated magnetic field data to match FPI timestamps.
Phi has an extra time dimension. Extracted phi from the first time step.
Phi bins shape: (32,)
Theta bins shape: (16,)
Calculated pitch angles.
pitch_angles shape: (4666, 32, 16)
Distribution data is already in the desired shape.
Transposed Distribution Data Shape: (4666, 32, 16, 32)
Pitch Angles Expanded Shape: (4666, 32, 16, 1)
Input Data Shape with Pitch Angles: (4666, 32, 16, 33)
Reshaped Input Data Shape: (4666, 1, 32, 16, 33)
Target Data Shape: (4666, 32, 16, 32)
Fine-Tuning Training Data Shape: (3732, 1, 32, 16, 33)
Fine-Tuning Validation Data Shape: (934, 1, 32, 

In [32]:
fpi_dist_folder = r"C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_02\fpi"
fgm_folder = r"C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_02\fgm"
fpi_moms_folder = r"C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_02\moms"

In [33]:
matched_files = match_files(fpi_dist_folder, fgm_folder, fpi_moms_folder)

In [39]:
def fine_tune_model_on_multiple_cdf_sets(
    matched_files,
    fpi_dist_folder,
    fgm_folder,
    fpi_moms_folder,
    model,
    fine_tune_epochs=10,
    batch_size=32,
    checkpoint_dir='fine_tuned_models'
):
    """
    Fine-tune the pre-trained model on multiple matched CDF file sets.
    
    Parameters:
    - matched_files: List of tuples containing matched CDF file names.
    - fpi_dist_folder: Directory containing FPI Distribution CDF files.
    - fgm_folder: Directory containing FGM CDF files.
    - fpi_moms_folder: Directory containing FPI Moments CDF files.
    - model: Pre-trained Keras model to be fine-tuned.
    - fine_tune_epochs: Number of epochs for fine-tuning on each file set.
    - batch_size: Batch size for training.
    - checkpoint_dir: Directory to save the fine-tuned models.
    
    Returns:
    - None
    """
    # Ensure the checkpoint directory exists
    os.makedirs(checkpoint_dir, exist_ok=True)
    print(f"Checkpoint directory set to: {checkpoint_dir}")
    
    # Fine-tune on all matched file sets
    total_sets = len(matched_files)
    print(f"\nTotal matched file sets to fine-tune on: {total_sets}")
    
    for idx, (timestamp, fpi_dist_filename, fgm_filename, fpi_moms_filename) in enumerate(matched_files, start=0):
        print(f"\n--- Fine-Tuning on CDF File Set {idx}/{total_sets} ---")
        print(f"Timestamp: {timestamp}")
        print(f"FPI Distribution File: {fpi_dist_filename}")
        print(f"FGM File: {fgm_filename}")
        print(f"FPI Moments File: {fpi_moms_filename}")
        
        # Construct full file paths
        fpi_dist_filepath = os.path.join(fpi_dist_folder, fpi_dist_filename)
        fgm_filepath = os.path.join(fgm_folder, fgm_filename)
        fpi_moms_filepath = os.path.join(fpi_moms_folder, fpi_moms_filename)
        
        # Prepare data
        X_data, y_data = prepare_data(fgm_filepath, fpi_dist_filepath, fpi_moms_filepath)
        
        # Split into fine-tuning training and validation sets (e.g., 80% train, 20% val)
        total_time_steps = X_data.shape[0]
        n_finetune_train = int(0.8 * total_time_steps)
        k_finetune_val = total_time_steps - n_finetune_train
        
        X_finetune_train = X_data[:n_finetune_train]
        y_finetune_train = y_data[:n_finetune_train]
        
        X_finetune_val = X_data[n_finetune_train:]
        y_finetune_val = y_data[n_finetune_train:]
        
        print(f"Fine-Tuning Training Data Shape: {X_finetune_train.shape}")  # (n_finetune_train,1,32,16,33)
        print(f"Fine-Tuning Validation Data Shape: {X_finetune_val.shape}")  # (k_finetune_val,1,32,16,33)
        
        # Define callbacks
        early_stopping = EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True,
            verbose=1
        )
        
        # Define a unique checkpoint path for each set
        checkpoint_path = os.path.join(checkpoint_dir, f'fine_tuned_model_set_{idx}.h5')
        checkpoint = ModelCheckpoint(
            filepath=checkpoint_path,
            monitor='val_loss',
            save_best_only=True,
            save_weights_only=False,  # Ensures the entire model is saved
            mode='min',
            verbose=1
        )
        
        # Fine-Tune the Model on Current CDF File Set
        history = model.fit(
            X_finetune_train,
            y_finetune_train,
            epochs=fine_tune_epochs,
            batch_size=batch_size,
            shuffle=False,  # Preserve time dependency
            validation_data=(X_finetune_val, y_finetune_val),
            callbacks=[early_stopping, checkpoint]
        )
        
        print(f"Fine-tuning on CDF File Set {idx} completed.")
        print(f"Best model for set {idx} saved at: {checkpoint_path}")
        
        # Optionally, save the training history for later analysis
        history_path = os.path.join(checkpoint_dir, f'history_finetune_set_{idx}.npy')
        np.save(history_path, history.history)
        print(f"Training history saved at: {history_path}")
    
    print("\nAll fine-tuning steps completed.")


In [36]:
fine_tune_model_on_multiple_cdf_sets(
    matched_files=matched_files,
    fpi_dist_folder=fpi_dist_folder,
    fgm_folder=fgm_folder,
    fpi_moms_folder=fpi_moms_folder,
    model=modified_model,  # Ensure this refers to your loaded modified model
    fine_tune_epochs=10,    # Adjust as needed
    batch_size=32,          # Adjust as needed
    checkpoint_dir='fine_tuned_models_03_02'  # Adjust as needed
)


Checkpoint directory set to: fine_tuned_models_03_02

Total matched file sets to fine-tune on: 9

--- Fine-Tuning on CDF File Set 0/9 ---
Timestamp: 2024-03-02 08:36:23
FPI Distribution File: mms1_fpi_brst_l2_des-dist_20240302083623_v3.4.0.cdf
FGM File: mms1_fgm_brst_l2_20240302083623_v5.441.0.cdf
FPI Moments File: mms1_fpi_brst_l2_des-moms_20240302083623_v3.4.0.cdf
Loaded FGM data from C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_02\fgm\mms1_fgm_brst_l2_20240302083623_v5.441.0.cdf.
Loaded FPI Distribution data from C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_02\fpi\mms1_fpi_brst_l2_des-dist_20240302083623_v3.4.0.cdf.
Loaded FPI Moments data from C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_02\moms\mms1_fpi_brst_l2_des-moms_20240302083623_v3.4.0.cdf.
Interpolated magnetic field data to match FPI timestamps.
Phi has an extra time dimension. Extracted phi from the first time step.
Phi bins shape: (32,)
Theta bins shape: (16,)
Calc

C:\Users\ryane\anaconda3\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


50/50 [==============================] - ETA: 0s - loss: 3.1473e-36
Epoch 2: val_loss did not improve from 0.00000
50/50 [==============================] - 3s 53ms/step - loss: 3.1473e-36 - val_loss: 5.0483e-05
Epoch 3/10
49/50 [============================>.] - ETA: 0s - loss: 0.0000e+00
Epoch 3: val_loss improved from 0.00000 to 0.00000, saving model to fine_tuned_models_03_02\fine_tuned_model_set_0.h5
50/50 [==============================] - 3s 57ms/step - loss: 0.0000e+00 - val_loss: 1.2475e-21
Epoch 4/10
49/50 [============================>.] - ETA: 0s - loss: 0.0000e+00
Epoch 4: val_loss did not improve from 0.00000
50/50 [==============================] - 3s 52ms/step - loss: 0.0000e+00 - val_loss: 2.0994e-05
Epoch 5/10
50/50 [==============================] - ETA: 0s - loss: 0.0000e+00
Epoch 5: val_loss did not improve from 0.00000
50/50 [==============================] - 3s 57ms/step - loss: 0.0000e+00 - val_loss: 8.8902e-09
Epoch 6/10
50/50 [==============================] - 

125/125 [==============================] - ETA: 0s - loss: 2.4794e-13
Epoch 1: val_loss improved from inf to 0.00000, saving model to fine_tuned_models_03_02\fine_tuned_model_set_2.h5
125/125 [==============================] - 7s 57ms/step - loss: 2.4794e-13 - val_loss: 3.7930e-17
Epoch 2/10
125/125 [==============================] - ETA: 0s - loss: 6.8823e-08
Epoch 2: val_loss did not improve from 0.00000
125/125 [==============================] - 7s 52ms/step - loss: 6.8823e-08 - val_loss: 8.0459e-12
Epoch 3/10
124/125 [============================>.] - ETA: 0s - loss: 3.7430e-12
Epoch 3: val_loss did not improve from 0.00000
125/125 [==============================] - 7s 54ms/step - loss: 3.7131e-12 - val_loss: 6.5325e-13
Epoch 4/10
125/125 [==============================] - ETA: 0s - loss: 1.2253e-15
Epoch 4: val_loss did not improve from 0.00000
125/125 [==============================] - 7s 53ms/step - loss: 1.2253e-15 - val_loss: 1.9115e-13
Epoch 5/10
125/125 [====================

Epoch 5/10
125/125 [==============================] - ETA: 0s - loss: 1.4286e-30
Epoch 5: val_loss did not improve from 0.00000
125/125 [==============================] - 8s 62ms/step - loss: 1.4286e-30 - val_loss: 1.0275e-15
Epoch 6/10
125/125 [==============================] - ETA: 0s - loss: 2.4261e-36Restoring model weights from the end of the best epoch: 1.

Epoch 6: val_loss did not improve from 0.00000
125/125 [==============================] - 7s 60ms/step - loss: 2.4261e-36 - val_loss: 8.3290e-17
Epoch 6: early stopping
Fine-tuning on CDF File Set 4 completed.
Best model for set 4 saved at: fine_tuned_models_03_02\fine_tuned_model_set_4.h5
Training history saved at: fine_tuned_models_03_02\history_finetune_set_4.npy

--- Fine-Tuning on CDF File Set 5/9 ---
Timestamp: 2024-03-02 11:41:33
FPI Distribution File: mms1_fpi_brst_l2_des-dist_20240302114133_v3.4.0.cdf
FGM File: mms1_fgm_brst_l2_20240302114133_v5.441.0.cdf
FPI Moments File: mms1_fpi_brst_l2_des-moms_20240302114133_v3.4

Loaded FPI Distribution data from C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_02\fpi\mms1_fpi_brst_l2_des-dist_20240302114633_v3.4.0.cdf.
Loaded FPI Moments data from C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_02\moms\mms1_fpi_brst_l2_des-moms_20240302114633_v3.4.0.cdf.
Interpolated magnetic field data to match FPI timestamps.
Phi has an extra time dimension. Extracted phi from the first time step.
Phi bins shape: (32,)
Theta bins shape: (16,)
Calculated pitch angles.
pitch_angles shape: (5000, 32, 16)
Distribution data is already in the desired shape.
Transposed Distribution Data Shape: (5000, 32, 16, 32)
Pitch Angles Expanded Shape: (5000, 32, 16, 1)
Input Data Shape with Pitch Angles: (5000, 32, 16, 33)
Reshaped Input Data Shape: (5000, 1, 32, 16, 33)
Target Data Shape: (5000, 32, 16, 32)
Fine-Tuning Training Data Shape: (4000, 1, 32, 16, 33)
Fine-Tuning Validation Data Shape: (1000, 1, 32, 16, 33)
Epoch 1/10
125/125 [========================

125/125 [==============================] - 7s 54ms/step - loss: 3.8095e-32 - val_loss: 5.4408e-16
Fine-tuning on CDF File Set 8 completed.
Best model for set 8 saved at: fine_tuned_models_03_02\fine_tuned_model_set_8.h5
Training history saved at: fine_tuned_models_03_02\history_finetune_set_8.npy

All fine-tuning steps completed.


In [40]:
fpi_dist_folder = r"C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_04\fpi"
fgm_folder = r"C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_04\fgm"
fpi_moms_folder = r"C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_04\moms"

In [41]:
matched_files = match_files(fpi_dist_folder, fgm_folder, fpi_moms_folder)

matched_files

[(datetime.datetime(2024, 3, 4, 19, 0, 33),
  'mms1_fpi_brst_l2_des-dist_20240304190033_v3.4.0.cdf',
  'mms1_fgm_brst_l2_20240304190033_v5.441.0.cdf',
  'mms1_fpi_brst_l2_des-moms_20240304190033_v3.4.0.cdf'),
 (datetime.datetime(2024, 3, 4, 19, 4, 23),
  'mms1_fpi_brst_l2_des-dist_20240304190423_v3.4.0.cdf',
  'mms1_fgm_brst_l2_20240304190423_v5.441.0.cdf',
  'mms1_fpi_brst_l2_des-moms_20240304190423_v3.4.0.cdf'),
 (datetime.datetime(2024, 3, 4, 22, 36, 33),
  'mms1_fpi_brst_l2_des-dist_20240304223633_v3.4.0.cdf',
  'mms1_fgm_brst_l2_20240304223633_v5.441.0.cdf',
  'mms1_fpi_brst_l2_des-moms_20240304223633_v3.4.0.cdf'),
 (datetime.datetime(2024, 3, 4, 22, 39, 33),
  'mms1_fpi_brst_l2_des-dist_20240304223933_v3.4.0.cdf',
  'mms1_fgm_brst_l2_20240304223933_v5.441.0.cdf',
  'mms1_fpi_brst_l2_des-moms_20240304223933_v3.4.0.cdf'),
 (datetime.datetime(2024, 3, 4, 22, 42, 43),
  'mms1_fpi_brst_l2_des-dist_20240304224243_v3.4.0.cdf',
  'mms1_fgm_brst_l2_20240304224243_v5.441.0.cdf',
  'mms1_fp

In [42]:


# Fine-tune the modified model on all matched file sets
fine_tune_model_on_multiple_cdf_sets(
    matched_files=matched_files,
    fpi_dist_folder=fpi_dist_folder,
    fgm_folder=fgm_folder,
    fpi_moms_folder=fpi_moms_folder,
    model=modified_model,  # Ensure this refers to your loaded modified model
    fine_tune_epochs=10,    # Adjust as needed
    batch_size=32,          # Adjust as needed
    checkpoint_dir='fine_tuned_models_03_04'  # Adjust as needed
)

Checkpoint directory set to: fine_tuned_models_03_04

Total matched file sets to fine-tune on: 5

--- Fine-Tuning on CDF File Set 0/5 ---
Timestamp: 2024-03-04 19:00:33
FPI Distribution File: mms1_fpi_brst_l2_des-dist_20240304190033_v3.4.0.cdf
FGM File: mms1_fgm_brst_l2_20240304190033_v5.441.0.cdf
FPI Moments File: mms1_fpi_brst_l2_des-moms_20240304190033_v3.4.0.cdf
Loaded FGM data from C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_04\fgm\mms1_fgm_brst_l2_20240304190033_v5.441.0.cdf.
Loaded FPI Distribution data from C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_04\fpi\mms1_fpi_brst_l2_des-dist_20240304190033_v3.4.0.cdf.
Loaded FPI Moments data from C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_04\moms\mms1_fpi_brst_l2_des-moms_20240304190033_v3.4.0.cdf.
Interpolated magnetic field data to match FPI timestamps.
Phi has an extra time dimension. Extracted phi from the first time step.
Phi bins shape: (32,)
Theta bins shape: (16,)
Calc

199/200 [============================>.] - ETA: 0s - loss: 0.0000e+00
Epoch 10: val_loss did not improve from 0.00000
200/200 [==============================] - 10s 50ms/step - loss: 0.0000e+00 - val_loss: 1.0027e-24
Fine-tuning on CDF File Set 1 completed.
Best model for set 1 saved at: fine_tuned_models_03_04\fine_tuned_model_set_1.h5
Training history saved at: fine_tuned_models_03_04\history_finetune_set_1.npy

--- Fine-Tuning on CDF File Set 2/5 ---
Timestamp: 2024-03-04 22:36:33
FPI Distribution File: mms1_fpi_brst_l2_des-dist_20240304223633_v3.4.0.cdf
FGM File: mms1_fgm_brst_l2_20240304223633_v5.441.0.cdf
FPI Moments File: mms1_fpi_brst_l2_des-moms_20240304223633_v3.4.0.cdf
Loaded FGM data from C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_04\fgm\mms1_fgm_brst_l2_20240304223633_v5.441.0.cdf.
Loaded FPI Distribution data from C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_04\fpi\mms1_fpi_brst_l2_des-dist_20240304223633_v3.4.0.cdf.
Loaded FPI Mome

Fine-tuning on CDF File Set 3 completed.
Best model for set 3 saved at: fine_tuned_models_03_04\fine_tuned_model_set_3.h5
Training history saved at: fine_tuned_models_03_04\history_finetune_set_3.npy

--- Fine-Tuning on CDF File Set 4/5 ---
Timestamp: 2024-03-04 22:42:43
FPI Distribution File: mms1_fpi_brst_l2_des-dist_20240304224243_v3.4.0.cdf
FGM File: mms1_fgm_brst_l2_20240304224243_v5.441.0.cdf
FPI Moments File: mms1_fpi_brst_l2_des-moms_20240304224243_v3.4.0.cdf
Loaded FGM data from C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_04\fgm\mms1_fgm_brst_l2_20240304224243_v5.441.0.cdf.
Loaded FPI Distribution data from C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_04\fpi\mms1_fpi_brst_l2_des-dist_20240304224243_v3.4.0.cdf.
Loaded FPI Moments data from C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_04\moms\mms1_fpi_brst_l2_des-moms_20240304224243_v3.4.0.cdf.
Interpolated magnetic field data to match FPI timestamps.
Phi has an extra tim

In [44]:
fpi_dist_folder = r"C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_05\fpi"
fgm_folder = r"C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_05\fgm"
fpi_moms_folder = r"C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_05\moms"

In [46]:
matched_files = match_files(fpi_dist_folder, fgm_folder, fpi_moms_folder)

In [48]:
matched_files

[(datetime.datetime(2024, 3, 5, 8, 48, 23),
  'mms1_fpi_brst_l2_des-dist_20240305084823_v3.4.0.cdf',
  'mms1_fgm_brst_l2_20240305084823_v5.441.0.cdf',
  'mms1_fpi_brst_l2_des-moms_20240305084823_v3.4.0.cdf'),
 (datetime.datetime(2024, 3, 5, 11, 25, 3),
  'mms1_fpi_brst_l2_des-dist_20240305112503_v3.4.0.cdf',
  'mms1_fgm_brst_l2_20240305112503_v5.441.0.cdf',
  'mms1_fpi_brst_l2_des-moms_20240305112503_v3.4.0.cdf'),
 (datetime.datetime(2024, 3, 5, 11, 28, 43),
  'mms1_fpi_brst_l2_des-dist_20240305112843_v3.4.0.cdf',
  'mms1_fgm_brst_l2_20240305112843_v5.441.0.cdf',
  'mms1_fpi_brst_l2_des-moms_20240305112843_v3.4.0.cdf'),
 (datetime.datetime(2024, 3, 5, 11, 32, 13),
  'mms1_fpi_brst_l2_des-dist_20240305113213_v3.4.0.cdf',
  'mms1_fgm_brst_l2_20240305113213_v5.441.0.cdf',
  'mms1_fpi_brst_l2_des-moms_20240305113213_v3.4.0.cdf'),
 (datetime.datetime(2024, 3, 5, 11, 35, 43),
  'mms1_fpi_brst_l2_des-dist_20240305113543_v3.4.0.cdf',
  'mms1_fgm_brst_l2_20240305113543_v5.441.0.cdf',
  'mms1_fp

In [49]:
# Fine-tune the modified model on all matched file sets
fine_tune_model_on_multiple_cdf_sets(
    matched_files=matched_files,
    fpi_dist_folder=fpi_dist_folder,
    fgm_folder=fgm_folder,
    fpi_moms_folder=fpi_moms_folder,
    model=modified_model,  # Ensure this refers to your loaded modified model
    fine_tune_epochs=10,    # Adjust as needed
    batch_size=64,          # Adjust as needed
    checkpoint_dir='fine_tuned_models_03_05'  # Adjust as needed
)

Checkpoint directory set to: fine_tuned_models_03_05

Total matched file sets to fine-tune on: 18

--- Fine-Tuning on CDF File Set 0/18 ---
Timestamp: 2024-03-05 08:48:23
FPI Distribution File: mms1_fpi_brst_l2_des-dist_20240305084823_v3.4.0.cdf
FGM File: mms1_fgm_brst_l2_20240305084823_v5.441.0.cdf
FPI Moments File: mms1_fpi_brst_l2_des-moms_20240305084823_v3.4.0.cdf


FileNotFoundError: C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_05\fgm\mms1_fgm_brst_l2_20240305084823_v5.441.0.cdf not found

In [50]:
modified_model.save('model_till_05.h5')

C:\Users\ryane\anaconda3\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [51]:
fpi_dist_folder = r"C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_07\fpi"
fgm_folder = r"C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_07\fgm"
fpi_moms_folder = r"C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_07\moms"
matched_files = match_files(fpi_dist_folder, fgm_folder, fpi_moms_folder)

In [53]:
fine_tune_model_on_multiple_cdf_sets(
    matched_files=matched_files,
    fpi_dist_folder=fpi_dist_folder,
    fgm_folder=fgm_folder,
    fpi_moms_folder=fpi_moms_folder,
    model=modified_model,  # Ensure this refers to your loaded modified model
    fine_tune_epochs=10,    # Adjust as needed
    batch_size=64,          # Adjust as needed
    checkpoint_dir='fine_tuned_models_03_07'  # Adjust as needed
)

Checkpoint directory set to: fine_tuned_models_03_07

Total matched file sets to fine-tune on: 1

--- Fine-Tuning on CDF File Set 0/1 ---
Timestamp: 2024-03-07 15:21:43
FPI Distribution File: mms1_fpi_brst_l2_des-dist_20240307152143_v3.4.0.cdf
FGM File: mms1_fgm_brst_l2_20240307152143_v5.442.0.cdf
FPI Moments File: mms1_fpi_brst_l2_des-moms_20240307152143_v3.4.0.cdf
Loaded FGM data from C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_07\fgm\mms1_fgm_brst_l2_20240307152143_v5.442.0.cdf.
Loaded FPI Distribution data from C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_07\fpi\mms1_fpi_brst_l2_des-dist_20240307152143_v3.4.0.cdf.
Loaded FPI Moments data from C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_07\moms\mms1_fpi_brst_l2_des-moms_20240307152143_v3.4.0.cdf.
Interpolated magnetic field data to match FPI timestamps.
Phi has an extra time dimension. Extracted phi from the first time step.
Phi bins shape: (32,)
Theta bins shape: (16,)
Calc

In [54]:
fpi_dist_folder = r"C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_08\fpi"
fgm_folder = r"C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_08\fgm"
fpi_moms_folder = r"C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_08\moms"
matched_files = match_files(fpi_dist_folder, fgm_folder, fpi_moms_folder)

In [55]:
fine_tune_model_on_multiple_cdf_sets(
    matched_files=matched_files,
    fpi_dist_folder=fpi_dist_folder,
    fgm_folder=fgm_folder,
    fpi_moms_folder=fpi_moms_folder,
    model=modified_model,  # Ensure this refers to your loaded modified model
    fine_tune_epochs=10,    # Adjust as needed
    batch_size=64,          # Adjust as needed
    checkpoint_dir='fine_tuned_models_03_07'  # Adjust as needed
)

Checkpoint directory set to: fine_tuned_models_03_07

Total matched file sets to fine-tune on: 1

--- Fine-Tuning on CDF File Set 0/1 ---
Timestamp: 2024-03-08 22:38:33
FPI Distribution File: mms1_fpi_brst_l2_des-dist_20240308223833_v3.4.0.cdf
FGM File: mms1_fgm_brst_l2_20240308223833_v5.442.0.cdf
FPI Moments File: mms1_fpi_brst_l2_des-moms_20240308223833_v3.4.0.cdf
Loaded FGM data from C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_08\fgm\mms1_fgm_brst_l2_20240308223833_v5.442.0.cdf.
Loaded FPI Distribution data from C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_08\fpi\mms1_fpi_brst_l2_des-dist_20240308223833_v3.4.0.cdf.
Loaded FPI Moments data from C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_08\moms\mms1_fpi_brst_l2_des-moms_20240308223833_v3.4.0.cdf.
Interpolated magnetic field data to match FPI timestamps.
Phi has an extra time dimension. Extracted phi from the first time step.
Phi bins shape: (32,)
Theta bins shape: (16,)
Calc

In [56]:
fpi_dist_folder = r"C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_09\fpi"
fgm_folder = r"C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_09\fgm"
fpi_moms_folder = r"C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_09\moms"
matched_files = match_files(fpi_dist_folder, fgm_folder, fpi_moms_folder)

In [57]:
fine_tune_model_on_multiple_cdf_sets(
    matched_files=matched_files,
    fpi_dist_folder=fpi_dist_folder,
    fgm_folder=fgm_folder,
    fpi_moms_folder=fpi_moms_folder,
    model=modified_model,  # Ensure this refers to your loaded modified model
    fine_tune_epochs=5,    # Adjust as needed
    batch_size=128,          # Adjust as needed
    checkpoint_dir='fine_tuned_models_03_09'  # Adjust as needed
)

Checkpoint directory set to: fine_tuned_models_03_09

Total matched file sets to fine-tune on: 9

--- Fine-Tuning on CDF File Set 0/9 ---
Timestamp: 2024-03-09 01:27:53
FPI Distribution File: mms1_fpi_brst_l2_des-dist_20240309012753_v3.4.0.cdf
FGM File: mms1_fgm_brst_l2_20240309012753_v5.442.0.cdf
FPI Moments File: mms1_fpi_brst_l2_des-moms_20240309012753_v3.4.0.cdf
Loaded FGM data from C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_09\fgm\mms1_fgm_brst_l2_20240309012753_v5.442.0.cdf.
Loaded FPI Distribution data from C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_09\fpi\mms1_fpi_brst_l2_des-dist_20240309012753_v3.4.0.cdf.
Loaded FPI Moments data from C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_09\moms\mms1_fpi_brst_l2_des-moms_20240309012753_v3.4.0.cdf.
Interpolated magnetic field data to match FPI timestamps.
Phi has an extra time dimension. Extracted phi from the first time step.
Phi bins shape: (32,)
Theta bins shape: (16,)
Calc

46/46 [==============================] - 8s 172ms/step - loss: 0.0000e+00 - val_loss: 2.1927e-22
Epoch 4/5
46/46 [==============================] - ETA: 0s - loss: 0.0000e+00
Epoch 4: val_loss improved from 0.00000 to 0.00000, saving model to fine_tuned_models_03_09\fine_tuned_model_set_2.h5
46/46 [==============================] - 7s 161ms/step - loss: 0.0000e+00 - val_loss: 8.6979e-23
Epoch 5/5
46/46 [==============================] - ETA: 0s - loss: 7.6424e-30
Epoch 5: val_loss improved from 0.00000 to 0.00000, saving model to fine_tuned_models_03_09\fine_tuned_model_set_2.h5
46/46 [==============================] - 7s 162ms/step - loss: 7.6424e-30 - val_loss: 3.4503e-23
Fine-tuning on CDF File Set 2 completed.
Best model for set 2 saved at: fine_tuned_models_03_09\fine_tuned_model_set_2.h5
Training history saved at: fine_tuned_models_03_09\history_finetune_set_2.npy

--- Fine-Tuning on CDF File Set 3/9 ---
Timestamp: 2024-03-09 09:07:53
FPI Distribution File: mms1_fpi_brst_l2_des-d

Epoch 1/5
42/42 [==============================] - ETA: 0s - loss: 0.0000e+00
Epoch 1: val_loss improved from inf to 0.00000, saving model to fine_tuned_models_03_09\fine_tuned_model_set_5.h5
42/42 [==============================] - 7s 161ms/step - loss: 0.0000e+00 - val_loss: 1.0873e-24
Epoch 2/5
42/42 [==============================] - ETA: 0s - loss: 0.0000e+00
Epoch 2: val_loss improved from 0.00000 to 0.00000, saving model to fine_tuned_models_03_09\fine_tuned_model_set_5.h5
42/42 [==============================] - 7s 170ms/step - loss: 0.0000e+00 - val_loss: 4.6743e-25
Epoch 3/5
42/42 [==============================] - ETA: 0s - loss: 2.8668e-29
Epoch 3: val_loss improved from 0.00000 to 0.00000, saving model to fine_tuned_models_03_09\fine_tuned_model_set_5.h5
42/42 [==============================] - 7s 168ms/step - loss: 2.8668e-29 - val_loss: 2.0094e-25
Epoch 4/5
42/42 [==============================] - ETA: 0s - loss: 5.4953e-13
Epoch 4: val_loss did not improve from 0.00000


Calculated pitch angles.
pitch_angles shape: (8333, 32, 16)
Distribution data is already in the desired shape.
Transposed Distribution Data Shape: (8333, 32, 16, 32)
Pitch Angles Expanded Shape: (8333, 32, 16, 1)
Input Data Shape with Pitch Angles: (8333, 32, 16, 33)
Reshaped Input Data Shape: (8333, 1, 32, 16, 33)
Target Data Shape: (8333, 32, 16, 32)
Fine-Tuning Training Data Shape: (6666, 1, 32, 16, 33)
Fine-Tuning Validation Data Shape: (1667, 1, 32, 16, 33)
Epoch 1/5
53/53 [==============================] - ETA: 0s - loss: 0.0000e+00
Epoch 1: val_loss improved from inf to 0.00000, saving model to fine_tuned_models_03_09\fine_tuned_model_set_8.h5
53/53 [==============================] - 9s 165ms/step - loss: 0.0000e+00 - val_loss: 3.0707e-22
Epoch 2/5
53/53 [==============================] - ETA: 0s - loss: 0.0000e+00
Epoch 2: val_loss improved from 0.00000 to 0.00000, saving model to fine_tuned_models_03_09\fine_tuned_model_set_8.h5
53/53 [==============================] - 9s 166m

In [58]:
fpi_dist_folder = r"C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_10\fpi"
fgm_folder = r"C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_10\fgm"
fpi_moms_folder = r"C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_10\moms"
matched_files = match_files(fpi_dist_folder, fgm_folder, fpi_moms_folder)

In [59]:
fine_tune_model_on_multiple_cdf_sets(
    matched_files=matched_files,
    fpi_dist_folder=fpi_dist_folder,
    fgm_folder=fgm_folder,
    fpi_moms_folder=fpi_moms_folder,
    model=modified_model,  # Ensure this refers to your loaded modified model
    fine_tune_epochs=5,    # Adjust as needed
    batch_size=128,          # Adjust as needed
    checkpoint_dir='fine_tuned_models_03_10'  # Adjust as needed
)

Checkpoint directory set to: fine_tuned_models_03_10

Total matched file sets to fine-tune on: 3

--- Fine-Tuning on CDF File Set 0/3 ---
Timestamp: 2024-03-10 23:18:33
FPI Distribution File: mms1_fpi_brst_l2_des-dist_20240310231833_v3.4.0.cdf
FGM File: mms1_fgm_brst_l2_20240310231833_v5.442.0.cdf
FPI Moments File: mms1_fpi_brst_l2_des-moms_20240310231833_v3.4.0.cdf
Loaded FGM data from C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_10\fgm\mms1_fgm_brst_l2_20240310231833_v5.442.0.cdf.
Loaded FPI Distribution data from C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_10\fpi\mms1_fpi_brst_l2_des-dist_20240310231833_v3.4.0.cdf.
Loaded FPI Moments data from C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_10\moms\mms1_fpi_brst_l2_des-moms_20240310231833_v3.4.0.cdf.
Interpolated magnetic field data to match FPI timestamps.
Phi has an extra time dimension. Extracted phi from the first time step.
Phi bins shape: (32,)
Theta bins shape: (16,)
Calc

44/44 [==============================] - 7s 161ms/step - loss: 0.0000e+00 - val_loss: 1.4689e-28
Epoch 4/5
44/44 [==============================] - ETA: 0s - loss: 0.0000e+00
Epoch 4: val_loss improved from 0.00000 to 0.00000, saving model to fine_tuned_models_03_10\fine_tuned_model_set_2.h5
44/44 [==============================] - 7s 154ms/step - loss: 0.0000e+00 - val_loss: 6.0660e-29
Epoch 5/5
44/44 [==============================] - ETA: 0s - loss: 0.0000e+00
Epoch 5: val_loss improved from 0.00000 to 0.00000, saving model to fine_tuned_models_03_10\fine_tuned_model_set_2.h5
44/44 [==============================] - 7s 155ms/step - loss: 0.0000e+00 - val_loss: 2.5050e-29
Fine-tuning on CDF File Set 2 completed.
Best model for set 2 saved at: fine_tuned_models_03_10\fine_tuned_model_set_2.h5
Training history saved at: fine_tuned_models_03_10\history_finetune_set_2.npy

All fine-tuning steps completed.


In [60]:
modified_model.save("model_3_10.h5")

In [ ]:
# ----------------------------------
# 11. Save the Final Fine-Tuned Model
# ----------------------------------

# Define the path to save the final fine-tuned model
final_fine_tuned_model_path = r"C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\final_fine_tuned_model_no_lambda.h5"

print(f"\nSaving the final fine-tuned model to: {final_fine_tuned_model_path}")
modified_model.save(final_fine_tuned_model_path)
print("Final fine-tuned model saved successfully.\n")

# ----------------------------------
# 12. Verify the Final Fine-Tuned Model (Optional)
# ----------------------------------

print("Verifying the saved final fine-tuned model by loading it...")
try:
    loaded_final_model = tf.keras.models.load_model(final_fine_tuned_model_path)
    print("Final fine-tuned model loaded successfully.")
    loaded_final_model.summary()
except Exception as e:
    print(f"Error loading the final fine-tuned model: {e}")

# ----------------------------------
# 13. Perform a Test Prediction (Optional)
# ----------------------------------

# Example Test Prediction
print("\nPerforming a test prediction with dummy data...")

# Define the input shape based on the model's input
# (batch_size, 1, 32, 16, 33)
batch_size = 1
time_steps = 10  # Example value; adjust as needed
phi_bins = 32
theta_bins = 16
energy_levels = 32

# Create dummy input data
dummy_input = np.random.random((batch_size, 1, phi_bins, theta_bins, energy_levels + 1)).astype(np.float32)

# Perform a prediction
try:
    prediction = loaded_final_model.predict(dummy_input)
    print("Prediction successful. Output shape:", prediction.shape)
except Exception as e:
    print(f"Error during prediction: {e}")
